# ITA06 – Machine Learning Lab
## 20 Practical Programs

**How to use:** Run the experiments one by one from top to bottom. Most programs use scikit-learn, pandas and NumPy, which are already available in Google Colab. If a program asks for a file, upload it using the Colab Files panel.

---
# Experiment 1 – FIND-S Algorithm

**Aim:** To implement and demonstrate the FIND-S algorithm for finding the most specific hypothesis based on given training samples.

Run the code cell below to execute this experiment.

In [1]:
data = [
    (["Sunny", "Warm", "Normal", "Strong"], "Yes"),
    (["Sunny", "Warm", "High", "Strong"], "Yes"),
    (["Rainy", "Cold", "High", "Strong"], "No"),
    (["Sunny", "Warm", "High", "Weak"], "Yes")
]
h = ["Ø"] * len(data[0][0])
for attrs, label in data:
    if label == "Yes":
        for i, value in enumerate(attrs):
            if h[i] == "Ø":
                h[i] = value
            elif h[i] != value:
                h[i] = "?"
print("Most specific hypothesis:", h)

Most specific hypothesis: ['Sunny', 'Warm', '?', '?']


---
# Experiment 2 – Candidate-Elimination Algorithm

**Aim:** To implement Candidate-Elimination in Python using training examples stored in a CSV file and display the version space boundaries.

Run the code cell below to execute this experiment.

In [2]:
import csv

# PlayTennis-style training data: Outlook, Temperature, Humidity, Wind, Class
rows = [
 ["Sunny","Warm","Normal","Strong","Yes"],
 ["Sunny","Warm","High","Strong","Yes"],
 ["Rainy","Cold","High","Strong","No"],
 ["Sunny","Warm","High","Weak","Yes"]
]
with open("training.csv","w",newline="") as f:
    csv.writer(f).writerows(rows)

data = list(csv.reader(open("training.csv")))
n = len(data[0])-1
S = ["Ø"]*n
G = [["?"]*n]

def covers(h, x):
    return all(a=="?" or a==v for a,v in zip(h,x))

for row in data:
    x, label = row[:-1], row[-1]
    if label == "Yes":
        S = [v if h=="Ø" else (h if h==v else "?") for h,v in zip(S,x)]
        G = [g for g in G if covers(g,x)]
    else:
        newG=[]
        for g in G:
            if covers(g,x):
                for i in range(n):
                    if S[i] not in ("Ø","?") and S[i] != x[i]:
                        spec=g.copy(); spec[i]=S[i]
                        if covers(spec, x) is False:
                            newG.append(spec)
        G=newG
print("S boundary:", S)
print("G boundary:", G)

S boundary: ['Sunny', 'Warm', '?', '?']
G boundary: [['Sunny', '?', '?', '?'], ['?', 'Warm', '?', '?']]


---
# Experiment 3 – Decision Tree using ID3 Algorithm

**Aim:** To construct a decision tree using the ID3 algorithm and classify a new sample.

Run the code cell below to execute this experiment.

In [3]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text

data = pd.DataFrame({
 "Outlook":["Sunny","Sunny","Overcast","Rain","Rain","Rain","Overcast","Sunny","Sunny","Rain","Sunny","Overcast","Overcast","Rain"],
 "Temperature":["Hot","Hot","Hot","Mild","Cool","Cool","Cool","Mild","Cool","Mild","Mild","Mild","Hot","Mild"],
 "Humidity":["High","High","High","High","Normal","Normal","Normal","High","Normal","Normal","Normal","High","Normal","High"],
 "Wind":["Weak","Strong","Weak","Weak","Weak","Strong","Strong","Weak","Weak","Weak","Strong","Strong","Weak","Strong"],
 "Play":["No","No","Yes","Yes","Yes","No","Yes","No","Yes","Yes","Yes","Yes","Yes","No"]})
X=pd.get_dummies(data.drop(columns="Play"))
y=data["Play"]
model=DecisionTreeClassifier(criterion="entropy", random_state=0)
model.fit(X,y)
print(export_text(model, feature_names=list(X.columns)))
sample=pd.DataFrame([{"Outlook":"Sunny","Temperature":"Cool","Humidity":"High","Wind":"Strong"}])
sample=pd.get_dummies(sample).reindex(columns=X.columns,fill_value=0)
print("Prediction:",model.predict(sample)[0])

|--- Outlook_Overcast <= 0.50
|   |--- Humidity_Normal <= 0.50
|   |   |--- Outlook_Rain <= 0.50
|   |   |   |--- class: No
|   |   |--- Outlook_Rain >  0.50
|   |   |   |--- Wind_Strong <= 0.50
|   |   |   |   |--- class: Yes
|   |   |   |--- Wind_Strong >  0.50
|   |   |   |   |--- class: No
|   |--- Humidity_Normal >  0.50
|   |   |--- Wind_Weak <= 0.50
|   |   |   |--- Temperature_Cool <= 0.50
|   |   |   |   |--- class: Yes
|   |   |   |--- Temperature_Cool >  0.50
|   |   |   |   |--- class: No
|   |   |--- Wind_Weak >  0.50
|   |   |   |--- class: Yes
|--- Outlook_Overcast >  0.50
|   |--- class: Yes

Prediction: No


---
# Experiment 4 – Artificial Neural Network using Backpropagation

**Aim:** To build an Artificial Neural Network using the backpropagation algorithm and test it on a suitable dataset.

Run the code cell below to execute this experiment.

In [4]:
import numpy as np
np.random.seed(1)
X=np.array([[0,0],[0,1],[1,0],[1,1]],float)
y=np.array([[0],[1],[1],[0]],float)  # XOR
W1=np.random.randn(2,4); b1=np.zeros((1,4))
W2=np.random.randn(4,1); b2=np.zeros((1,1))
sig=lambda z: 1/(1+np.exp(-z))
for epoch in range(15000):
    h=sig(X@W1+b1); o=sig(h@W2+b2)
    d2=(o-y)*o*(1-o)
    d1=(d2@W2.T)*h*(1-h)
    W2-=0.8*h.T@d2/len(X); b2-=0.8*d2.mean(axis=0,keepdims=True)
    W1-=0.8*X.T@d1/len(X); b1-=0.8*d1.mean(axis=0,keepdims=True)
print("Predicted XOR:",(sig(sig(X@W1+b1)@W2+b2)>=.5).astype(int).ravel())

Predicted XOR: [0 1 1 0]


---
# Experiment 5 – K-Nearest Neighbours (K-NN)

**Aim:** To implement the K-Nearest Neighbours classification algorithm in Python.

Run the code cell below to execute this experiment.

In [5]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
X,y=load_iris(return_X_y=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
model=KNeighborsClassifier(n_neighbors=5)
model.fit(Xtr,ytr)
pred=model.predict(Xte)
print("Accuracy:",round(accuracy_score(yte,pred)*100,2),"%")
print("First five predictions:",pred[:5])

Accuracy: 100.0 %
First five predictions: [0 2 1 1 0]


---
# Experiment 6 – Naïve Bayes Classification with Confusion Matrix

**Aim:** To implement the Naïve Bayes algorithm and evaluate its performance using a confusion matrix and accuracy.

Run the code cell below to execute this experiment.

In [6]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, accuracy_score
X,y=load_iris(return_X_y=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
model=GaussianNB().fit(Xtr,ytr)
pred=model.predict(Xte)
print("Confusion Matrix:\n",confusion_matrix(yte,pred))
print("Accuracy:",round(accuracy_score(yte,pred)*100,2),"%")

Confusion Matrix:
 [[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]
Accuracy: 96.67 %


---
# Experiment 7 – Logistic Regression Classification

**Aim:** To implement Logistic Regression for a classification problem using Python.

Run the code cell below to execute this experiment.

In [7]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
X,y=load_breast_cancer(return_X_y=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=3000))
model.fit(Xtr,ytr)
pred=model.predict(Xte)
print("Accuracy:",round(accuracy_score(yte,pred)*100,2),"%")
print("First five predictions:",pred[:5])

Accuracy: 98.25 %
First five predictions: [0 1 0 1 0]


---
# Experiment 8 – Linear Regression

**Aim:** To implement Linear Regression in Python to predict a continuous numerical value.

Run the code cell below to execute this experiment.

In [8]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
X=np.array([[1],[2],[3],[4],[5],[6],[7],[8],[9],[10]])
y=np.array([3,5,7,9,11,13,15,17,19,21])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=1)
model=LinearRegression().fit(Xtr,ytr)
pred=model.predict(Xte)
print("Slope:",model.coef_[0])
print("Intercept:",model.intercept_)
print("Predictions:",pred)
print("MSE:",mean_squared_error(yte,pred))
print("R2:",r2_score(yte,pred))

Slope: 2.0
Intercept: 1.0
Predictions: [ 7. 21.]
MSE: 0.0
R2: 1.0


---
# Experiment 9 – Linear and Polynomial Regression Comparison

**Aim:** To compare Linear Regression and Polynomial Regression using Python.

Run the code cell below to execute this experiment.

In [9]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
rng=np.random.default_rng(10)
X=np.linspace(-3,3,100).reshape(-1,1)
y=2+X.ravel()**2+rng.normal(0,.5,100)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
linear=LinearRegression().fit(Xtr,ytr)
poly=PolynomialFeatures(degree=2)
Xtrp=poly.fit_transform(Xtr); Xtep=poly.transform(Xte)
quadratic=LinearRegression().fit(Xtrp,ytr)
print("Linear MSE:",mean_squared_error(yte,linear.predict(Xte)))
print("Polynomial MSE:",mean_squared_error(yte,quadratic.predict(Xtep)))

Linear MSE: 6.23993474609639
Polynomial MSE: 0.12495753659337731


---
# Experiment 10 – Expectation-Maximization Algorithm

**Aim:** To implement the Expectation-Maximization (EM) algorithm using a Gaussian Mixture Model.

Run the code cell below to execute this experiment.

In [10]:
import numpy as np
from sklearn.mixture import GaussianMixture
rng=np.random.default_rng(42)
X=np.r_[rng.normal(-2,.5,(100,1)),rng.normal(2,.7,(100,1))]
model=GaussianMixture(n_components=2,random_state=42)
model.fit(X)
print("Means:",np.round(model.means_.ravel(),2))
print("Weights:",np.round(model.weights_,2))
print("Converged:",model.converged_)
print("Iterations:",model.n_iter_)

Means: [-2.03  1.99]
Weights: [0.5 0.5]
Converged: True
Iterations: 2


---
# Experiment 11 – Credit Score Classification

**Aim:** To classify credit score categories using a machine learning classification algorithm.

Run the code cell below to execute this experiment.

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
data=pd.DataFrame({
 "income":[25,30,45,60,80,90,35,55,75,100,40,65],
 "debt":[18,20,15,12,8,5,25,14,9,4,22,10],
 "late_payments":[4,3,2,1,0,0,5,1,0,0,3,1],
 "score":["Low","Low","Medium","Medium","High","High","Low","Medium","High","High","Low","Medium"]})
X=data.drop(columns="score"); y=data["score"]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=1,stratify=y)
m=RandomForestClassifier(n_estimators=100,random_state=1).fit(Xtr,ytr)
p=m.predict(Xte)
print("Predictions:",p)
print("Accuracy:",accuracy_score(yte,p))

Predictions: ['Medium' 'High' 'Low']
Accuracy: 1.0


---
# Experiment 12 – Iris Flower Classification using KNN

**Aim:** To classify Iris flowers into Setosa, Versicolor or Virginica using K-Nearest Neighbours.

Run the code cell below to execute this experiment.

In [12]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
iris=load_iris()
Xtr,Xte,ytr,yte=train_test_split(iris.data,iris.target,test_size=.2,
    random_state=42,stratify=iris.target)
model=make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=5))
model.fit(Xtr,ytr)
pred=model.predict(Xte)
print("Accuracy:",round(accuracy_score(yte,pred)*100,2),"%")
print("Predicted species:",iris.target_names[model.predict([[5.1,3.5,1.4,.2]])[0]])

Accuracy: 93.33 %
Predicted species: setosa


---
# Experiment 13 – Car Price Prediction

**Aim:** To build a machine learning model to predict car prices from selected car features.

Run the code cell below to execute this experiment.

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
data=pd.DataFrame({
 "age":[1,2,3,4,5,6,7,8,2,4,6,3],
 "mileage":[10,20,30,40,50,60,70,80,18,42,65,28],
 "engine":[2.0,2.0,1.8,1.6,1.5,1.5,1.2,1.0,2.2,1.8,1.4,2.0],
 "price":[18,16,14,12,10,8,6,4,17,11,7,15]})
X=data[["age","mileage","engine"]]; y=data["price"]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=2)
m=LinearRegression().fit(Xtr,ytr)
p=m.predict(Xte)
print("Predicted prices (in lakh):",p.round(2))
print("Mean Absolute Error:",round(mean_absolute_error(yte,p),2))

Predicted prices (in lakh): [11.86  8.33 10.07]
Mean Absolute Error: 0.42


---
# Experiment 14 – House Price Prediction

**Aim:** To predict house prices using a suitable machine learning regression algorithm.

Run the code cell below to execute this experiment.

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
df=pd.DataFrame({
 "area":[800,1000,1200,1500,1800,2000,900,1300,1600,2200,1100,1900],
 "bedrooms":[2,2,3,3,3,4,2,3,3,4,2,4],
 "age":[20,15,10,8,5,2,25,12,6,1,18,4],
 "price":[35,42,55,68,75,95,32,58,72,110,45,88]})
X=df[["area","bedrooms","age"]]; y=df["price"]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42)
m=RandomForestRegressor(n_estimators=100,random_state=42).fit(Xtr,ytr)
p=m.predict(Xte)
print("Predicted prices:",p.round(2))
print("MAE:",round(mean_absolute_error(yte,p),2))
print("R2:",round(r2_score(yte,p),2))

Predicted prices: [44.96 89.89 38.45]
MAE: 7.87
R2: 0.87


---
# Experiment 15 – Iris Classification using Naïve Bayes

**Aim:** To implement Iris flower classification using the Gaussian Naïve Bayes classifier.

Run the code cell below to execute this experiment.

In [15]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix
iris=load_iris()
Xtr,Xte,ytr,yte=train_test_split(iris.data,iris.target,test_size=.2,
    random_state=42,stratify=iris.target)
model=GaussianNB().fit(Xtr,ytr)
pred=model.predict(Xte)
print("Confusion Matrix:\n",confusion_matrix(yte,pred))
print("Accuracy:",round(accuracy_score(yte,pred)*100,2),"%")
print("First five predicted classes:",pred[:5])

Confusion Matrix:
 [[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]
Accuracy: 96.67 %
First five predicted classes: [0 2 1 1 0]


---
# Experiment 16 – Comparison of Classification Algorithms

**Aim:** To compare different classification algorithms and evaluate their performance on the same dataset.

Run the code cell below to execute this experiment.

In [16]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
X,y=load_breast_cancer(return_X_y=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
models={
 "KNN":make_pipeline(StandardScaler(),KNeighborsClassifier(5)),
 "Naive Bayes":GaussianNB(),
 "Logistic Regression":make_pipeline(StandardScaler(),LogisticRegression(max_iter=3000)),
 "Decision Tree":DecisionTreeClassifier(random_state=42)}
for name,m in models.items():
    m.fit(Xtr,ytr)
    print(name,":",round(accuracy_score(yte,m.predict(Xte))*100,2),"%")

KNN : 95.61 %
Naive Bayes : 93.86 %
Logistic Regression : 98.25 %
Decision Tree : 91.23 %


---
# Experiment 17 – Mobile Price Prediction

**Aim:** To predict the mobile phone price range using machine learning based on its specifications.

Run the code cell below to execute this experiment.

In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
df=pd.DataFrame({
 "ram":[2,3,4,6,8,12,2,4,6,8,3,10],
 "battery":[2500,3000,3500,4000,4500,5000,2800,3600,4200,4800,3200,5000],
 "memory":[16,32,64,128,128,256,16,64,128,256,32,256],
 "price_range":[0,0,1,2,3,3,0,1,2,3,0,3]})
X=df.drop(columns="price_range"); y=df["price_range"]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.34,random_state=42,stratify=y)
m=RandomForestClassifier(n_estimators=100,random_state=42).fit(Xtr,ytr)
p=m.predict(Xte)
print("Predicted price ranges:",p)
print("Accuracy:",accuracy_score(yte,p))

Predicted price ranges: [3 2 0 0 1]
Accuracy: 1.0


---
# Experiment 18 – Perceptron-based Iris Classification

**Aim:** To implement a Perceptron classifier for Iris flower classification.

Run the code cell below to execute this experiment.

In [18]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score
iris=load_iris()
# Binary classification: Setosa (0) and Versicolor (1)
mask=iris.target<2
X=iris.data[mask][:,[0,2]]
y=iris.target[mask]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
model=Perceptron(max_iter=1000,eta0=.1,random_state=42)
model.fit(Xtr,ytr)
pred=model.predict(Xte)
print("Accuracy:",round(accuracy_score(yte,pred)*100,2),"%")
print("Predictions:",pred[:10])

Accuracy: 100.0 %
Predictions: [1 1 1 1 0 0 0 1 0 0]


---
# Experiment 19 – Naïve Bayes Classification for Bank Loan Prediction

**Aim:** To predict whether a bank loan applicant is likely to be approved using a Naïve Bayes classifier.

Run the code cell below to execute this experiment.

In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix
df=pd.DataFrame({
 "income":[25,30,45,60,80,90,35,55,75,100,40,65,28,85,50,70],
 "loan_amount":[10,15,20,25,30,35,18,22,28,40,12,24,16,32,20,27],
 "credit_history":[0,0,1,1,1,1,0,1,1,1,0,1,0,1,1,1],
 "approved":[0,0,1,1,1,1,0,1,1,1,0,1,0,1,1,1]})
X=df.drop(columns="approved"); y=df["approved"]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=1,stratify=y)
m=GaussianNB().fit(Xtr,ytr)
p=m.predict(Xte)
print("Predictions (0=Rejected, 1=Approved):",p)
print("Confusion Matrix:\n",confusion_matrix(yte,p))
print("Accuracy:",accuracy_score(yte,p))

Predictions (0=Rejected, 1=Approved): [1 1 1 0]
Confusion Matrix:
 [[1 0]
 [0 3]]
Accuracy: 1.0


---
# Experiment 20 – Future Sales Prediction

**Aim:** To predict future sales using a suitable machine learning regression algorithm.

Run the code cell below to execute this experiment.

In [20]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
# Monthly sales (in units), ordered by time
months=np.arange(1,13).reshape(-1,1)
sales=np.array([120,135,128,150,160,158,175,180,190,205,210,225])
Xtr,Xte=months[:9],months[9:]
ytr,yte=sales[:9],sales[9:]
model=LinearRegression().fit(Xtr,ytr)
pred=model.predict(Xte)
print("Predicted sales for months 10-12:",pred.round(2))
print("Actual sales:",yte)
print("Mean Absolute Error:",round(mean_absolute_error(yte,pred),2))

Predicted sales for months 10-12: [198.19 206.81 215.43]
Actual sales: [205 210 225]
Mean Absolute Error: 6.52


## End of practicals
Save a copy of this notebook in your Google Drive after opening it in Colab.